# Notebook 1 (Revised): Dataset Preparation and Archetype Discovery

This notebook replaces the discovery section of the original draft before any classifier is trained.
The original run found five numeric clusters, but one cluster contained most labeled complaints and
its keywords were dominated by stopwords. That output should be treated as a preliminary baseline,
not as final training labels.

This revision adds:

1. Auditable scam-corpus filtering and exact deduplication
2. Category-aware entity normalization instead of replacing every brand with `company`
3. Cached sentence embeddings
4. A small HDBSCAN parameter benchmark
5. Stopword-aware BERTopic representations
6. A manual topic-review gate before train/validation/test splits are finalized


## 1. Install dependencies


In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "scikit-learn>=1.5,<2" \
  "matplotlib>=3.8,<4" "sentence-transformers>=3.4,<6" "umap-learn>=0.5.7,<0.6" \
  "hdbscan>=0.8.40,<0.9" "bertopic>=0.17,<0.19" "gensim>=4.3,<5" ipywidgets


## 2. Imports, paths, and reproducibility


In [ ]:
import json, os, random, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import normalize_complaint, is_scam_candidate

with open(PROJECT_ROOT / "configs/project_config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

RAW = PROJECT_ROOT / CONFIG["raw_csv"]
OUT = PROJECT_ROOT / CONFIG["processed_dir"]
OUT.mkdir(parents=True, exist_ok=True)

COL_NARR = "Consumer complaint narrative"
COL_TAGS = "Tags"
COL_ID = "Complaint ID"
COL_SUB = "Sub-product"
COL_PROD = "Product"
OLDER_TAG = "Older American"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW)


## 3. Load and validate the CFPB export


In [ ]:
if not RAW.exists():
    raise FileNotFoundError(
        f"Place the CFPB CSV at {RAW}. The dataset is not included in this repository."
    )

df = pd.read_csv(RAW, low_memory=False)
required = {COL_NARR, COL_TAGS, COL_ID, COL_SUB, COL_PROD}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = df[df[COL_NARR].notna()].copy()
df[COL_NARR] = df[COL_NARR].astype(str)
print(f"Rows with a narrative: {len(df):,}")
print(df[[COL_PROD, COL_SUB]].value_counts().head(10))


## 4. Initial exploration


In [ ]:
df["raw_word_count"] = df[COL_NARR].str.split().str.len()
df["is_older"] = df[COL_TAGS].astype(str).str.contains(OLDER_TAG, na=False)

print(df["raw_word_count"].describe().round(1))
print(f"Older-consumer share: {df['is_older'].mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["raw_word_count"].clip(upper=600).hist(bins=50, ax=axes[0])
axes[0].set(title="Narrative length (clipped at 600 words)", xlabel="Words", ylabel="Complaints")
df[COL_SUB].value_counts().head(10).sort_values().plot.barh(ax=axes[1])
axes[1].set(title="Top sub-products", xlabel="Complaints")
plt.tight_layout()
plt.show()


## 5. Normalize, audit, and deduplicate

The attached CSV contains 25,777 complaint narratives and appears to be the team's already-selected
working corpus from Milestone 2. A narrow keyword rule is **not** used as the final inclusion filter:
an audit showed that it misses clear scams described without words such as `fraud` or `scam`
(for example, fake rental listings, advance-fee winnings, and marketplace overpayment schemes).

The regex-based scam indicator is therefore retained only as an audit feature. Final discovery keeps
the full cleaned corpus, while BERTopic noise handling and the manual review gate are used to reject
ordinary service-dispute or incoherent topics.

Entity names are normalized into categories such as `payment_app`, `bank`, and `crypto_exchange`.
Exact normalized duplicates are removed before splitting to prevent leakage.


In [ ]:
df["clean_text"] = df[COL_NARR].map(normalize_complaint)
df["word_count"] = df["clean_text"].str.split().str.len()

before = len(df)
df = df[df["word_count"] >= CONFIG["min_words"]].copy()
print(f"Removed {before - len(df):,} very short narratives.")

# Exact normalized duplicates can leak across splits and inflate evaluation.
before = len(df)
df = df.drop_duplicates(subset=["clean_text"]).copy()
print(f"Removed {before - len(df):,} exact duplicate narratives.")

# Keep this as an audit signal rather than treating a brittle keyword rule as ground truth.
df["regex_scam_flag"] = df["clean_text"].map(is_scam_candidate)
print(f"Rows matched by the high-recall regex audit: {df['regex_scam_flag'].mean():.1%}")

if CONFIG.get("filter_scam_candidates", False):
    excluded = df.loc[~df["regex_scam_flag"]].copy()
    df = df.loc[df["regex_scam_flag"]].copy()
    excluded.sample(min(100, len(excluded)), random_state=SEED).to_csv(
        OUT / "excluded_filter_audit_sample.csv", index=False
    )
    print(
        "WARNING: regex filtering is enabled. Review excluded_filter_audit_sample.csv "
        "because keyword filtering can remove genuine scams."
    )
else:
    print("Keeping the complete cleaned working corpus for discovery.")

max_words = CONFIG["max_words_discovery"]
df["clean_text"] = df["clean_text"].map(lambda x: " ".join(x.split()[:max_words]))

df.to_csv(OUT / "complaints_clean.csv.gz", index=False, compression="gzip")
print("Saved:", OUT / "complaints_clean.csv.gz")


## 6. Encode narratives once and cache the embeddings

Sentence embeddings are the expensive part of the discovery pipeline. Caching them lets us compare
several clustering settings without recomputing the Transformer output.

The first run downloads `sentence-transformers/all-MiniLM-L6-v2` from Hugging Face. Run this cell in
an environment with Hugging Face access, then keep `minilm_embeddings.npy` for later experiments.
Do not commit the embedding file to GitHub.


In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_PATH = OUT / "minilm_embeddings.npy"
embedder = SentenceTransformer(CONFIG["embedding_model"])
docs = df["clean_text"].tolist()

if EMBED_PATH.exists():
    embeddings = np.load(EMBED_PATH)
    if len(embeddings) != len(docs):
        raise ValueError("Cached embedding count does not match the cleaned dataset. Delete the cache and rerun.")
    print("Loaded cached embeddings:", embeddings.shape)
else:
    embeddings = embedder.encode(
        docs,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(EMBED_PATH, embeddings)
    print("Saved embeddings:", embeddings.shape)


## 7. Preliminary experiment: benchmark clustering settings

We compare a small, documented grid rather than accepting the first HDBSCAN result. The benchmark
records DBCV, number of clusters, noise rate, and the share of the largest cluster. A model with a
high DBCV but one giant cluster is not useful for downstream classification.


In [ ]:
from hdbscan import HDBSCAN
from umap import UMAP

reducer = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)
reduced = reducer.fit_transform(embeddings)
np.save(OUT / "umap_embeddings.npy", reduced)

GRID = [
    {"min_cluster_size": 75,  "min_samples": 10, "cluster_selection_method": "leaf"},
    {"min_cluster_size": 100, "min_samples": 15, "cluster_selection_method": "leaf"},
    {"min_cluster_size": 150, "min_samples": 15, "cluster_selection_method": "leaf"},
    {"min_cluster_size": 100, "min_samples": 10, "cluster_selection_method": "eom"},
    {"min_cluster_size": 200, "min_samples": 20, "cluster_selection_method": "eom"},
    {"min_cluster_size": 300, "min_samples": 30, "cluster_selection_method": "eom"},
]

rows = []
for params in GRID:
    clusterer = HDBSCAN(
        metric="euclidean",
        prediction_data=True,
        gen_min_span_tree=True,
        **params,
    )
    labels = clusterer.fit_predict(reduced)
    assigned = labels[labels != -1]
    counts = pd.Series(assigned).value_counts()
    n_clusters = int(counts.size)
    noise_rate = float((labels == -1).mean())
    largest_share = float(counts.iloc[0] / len(assigned)) if len(assigned) else 1.0
    dbcv = float(clusterer.relative_validity_)
    score = dbcv - 0.15 * noise_rate - 0.50 * max(0.0, largest_share - 0.45)
    rows.append({**params, "n_clusters": n_clusters, "noise_rate": noise_rate,
                 "largest_cluster_share": largest_share, "dbcv": dbcv, "selection_score": score})

benchmark = pd.DataFrame(rows).sort_values("selection_score", ascending=False)
benchmark.to_csv(OUT / "clustering_benchmark.csv", index=False)
benchmark


In [ ]:
# Prefer interpretable candidates, then use the documented score.
feasible = benchmark.query("4 <= n_clusters <= 30 and largest_cluster_share <= 0.60")
selected = (feasible if len(feasible) else benchmark).iloc[0].to_dict()
SELECTED_PARAMS = {
    "min_cluster_size": int(selected["min_cluster_size"]),
    "min_samples": int(selected["min_samples"]),
    "cluster_selection_method": selected["cluster_selection_method"],
}
print("Selected HDBSCAN settings:", SELECTED_PARAMS)
print("This automatic choice is provisional; the representative-document review below is decisive.")


## 8. Fit the final BERTopic model

Stopword removal belongs in BERTopic's topic representation layer, not in the sentence embeddings.
The vectorizer uses English stopwords and unigrams/bigrams, while `KeyBERTInspired` improves the
readability of the final keywords.


In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.90,
)
representation_model = KeyBERTInspired()
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

final_umap = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)
final_hdbscan = HDBSCAN(
    metric="euclidean",
    prediction_data=True,
    gen_min_span_tree=True,
    **SELECTED_PARAMS,
)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=final_umap,
    hdbscan_model=final_hdbscan,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)
df["archetype"] = topics
df["cluster_confidence"] = final_hdbscan.probabilities_

print(topic_model.get_topic_info().head(20))
print(f"Noise rate: {(df['archetype'] == -1).mean():.1%}")

DISCOVERY_MODEL_DIR = PROJECT_ROOT / "models" / "bertopic"
DISCOVERY_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
try:
    topic_model.save(str(DISCOVERY_MODEL_DIR), serialization="safetensors", save_ctfidf=True)
except TypeError:
    topic_model.save(str(DISCOVERY_MODEL_DIR))
print("Saved BERTopic model to", DISCOVERY_MODEL_DIR)


## 9. Validate and review the discovered topics


In [ ]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# Geometric cluster quality.
dbcv = float(final_hdbscan.relative_validity_)

# Coherence using unigram terms from the topic representation.
topic_words = []
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    words = []
    for term, _ in topic_model.get_topic(topic_id):
        words.extend(part for part in term.split() if part.isalpha())
        if len(words) >= 10:
            break
    topic_words.append(words[:10])

tokenized = [text.split() for text in docs]
dictionary = Dictionary(tokenized)
coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized,
    dictionary=dictionary,
    coherence="c_v",
)
coherence = float(coherence_model.get_coherence())
print(f"DBCV: {dbcv:.3f}")
print(f"Mean C_v coherence: {coherence:.3f}")


In [ ]:
review_rows = []
info = topic_model.get_topic_info()
for _, row in info[info["Topic"] != -1].iterrows():
    topic_id = int(row["Topic"])
    keywords = ", ".join(term for term, _ in topic_model.get_topic(topic_id)[:10])
    representatives = topic_model.get_representative_docs(topic_id) or []
    excerpts = [text[:500].replace("\n", " ") for text in representatives[:3]]
    review_rows.append({
        "topic_id": topic_id,
        "count": int(row["Count"]),
        "keywords": keywords,
        "representative_1": excerpts[0] if len(excerpts) > 0 else "",
        "representative_2": excerpts[1] if len(excerpts) > 1 else "",
        "representative_3": excerpts[2] if len(excerpts) > 2 else "",
        "proposed_label": "",
        "keep_topic": "yes",
        "review_notes": "",
    })

topic_review = pd.DataFrame(review_rows)
topic_review.to_csv(OUT / "topic_review_template.csv", index=False)
topic_review[["topic_id", "count", "keywords"]]


### Manual review gate

Open `data/processed/topic_review_template.csv`. For every topic, read the keywords and at least
three representative complaints, enter a concise human-readable label, and mark incoherent topics
as `keep_topic = no`.

The dictionary below can be filled directly after the review. Until it is filled, the notebook saves
provisional numeric labels so the pipeline can be tested, but those labels should not be used for the
final reported classifier results.


In [ ]:
# Example only: replace with labels agreed by the team after reviewing the CSV.
TOPIC_LABELS = {
    # 0: "Unauthorized payment-app transfer",
    # 1: "Counterfeit-check or overpayment scam",
}
REJECTED_TOPICS = {
    # Add incoherent topic IDs here, for example: 7,
}

missing_labels = sorted(set(df.loc[df["archetype"] != -1, "archetype"]) - set(TOPIC_LABELS))
labels_are_provisional = bool(missing_labels)
if labels_are_provisional:
    print("WARNING: human labels are still missing for topic IDs:", missing_labels)

df["archetype_label"] = df["archetype"].map(TOPIC_LABELS)
df["archetype_label"] = df["archetype_label"].fillna(
    df["archetype"].map(lambda value: f"topic_{value}" if value != -1 else "noise")
)
df["labels_are_provisional"] = labels_are_provisional


## 10. Build leakage-resistant train/validation/test splits


In [ ]:
from sklearn.model_selection import train_test_split

labeled = df[(df["archetype"] != -1) & (~df["archetype"].isin(REJECTED_TOPICS))].copy()
class_counts = labeled["archetype_label"].value_counts()
if (class_counts < 10).any():
    small_classes = class_counts[class_counts < 10]
    raise ValueError(f"Classes with fewer than 10 records cannot be safely stratified: {small_classes.to_dict()}")

train_val, test = train_test_split(
    labeled,
    test_size=0.10,
    stratify=labeled["archetype_label"],
    random_state=SEED,
)
train, val = train_test_split(
    train_val,
    test_size=0.10 / 0.90,
    stratify=train_val["archetype_label"],
    random_state=SEED,
)

for name, split in {"train": train, "val": val, "test": test}.items():
    split.to_csv(OUT / f"split_{name}.csv.gz", index=False, compression="gzip")
    print(name, len(split), split["archetype_label"].nunique())

labeled.to_csv(OUT / "discovery_output.csv.gz", index=False, compression="gzip")

summary = {
    "rows_after_cleaning": int(len(df)),
    "labeled_rows": int(len(labeled)),
    "n_archetypes": int(labeled["archetype_label"].nunique()),
    "noise_rate": float((df["archetype"] == -1).mean()),
    "dbcv": dbcv,
    "c_v_coherence": coherence,
    "labels_are_provisional": labels_are_provisional,
    "selected_hdbscan": SELECTED_PARAMS,
}
with open(OUT / "discovery_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
summary


## Stop here before final classifier training

Proceed to Notebook 2 for a preliminary software test even if labels are provisional. Before the
final experiment, complete the topic-review CSV, update `TOPIC_LABELS`, rerun this notebook, and
confirm that no single class dominates the labeled data without a defensible interpretation.
